# <b>1 <span style='color:#00008B'>|</span>  Pré Processamento</b>

## <b>1.1 <span style='color:#2ae4f5'>|</span>  Imports de Bibliotecas</b>

In [ ]:
%pip install pandas numpy matplotlib seaborn sklearn xgboost lightgbm optuna catboost

In [2]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from utils import ks_statistic, select_threshold_by_alert, simulate_financial_impact
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, TimeSeriesSplit, cross_val_predict
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve,accuracy_score, precision_score, recall_score, f1_score, classification_report,average_precision_score,roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.utils.validation import validate_data
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.pipeline import Pipeline as ImbPipeline

import warnings
warnings.filterwarnings('ignore')

## <b>1.2 <span style='color:#2ae4f5'>|</span>  Carregamento do Dataset</b>

In [3]:
df = pd.read_csv("../datasets/base_antifraude.gz", compression="gzip", sep="\t")
PROFIT_MARGIN = 0.05  # 5% de margem nos bons clientes aprovados
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS = 5

## <b> 1.3 <span style='color:#2ae4f5'>|</span>  Pipeline de Pré-processamento</b>

### <b> 1.3.1 <span style='color:#2ae4f5'>|</span>  Separação das bases de Treino e Teste</b>

Para evitarmos Data Leakage, vamos seguir com a separacão de teste e treino antes da aplicação do pipeline de processamento, evitando que o modelo apresente desempenho artificialmente elevado na fase de validação/teste, mas falhe em generalizar para dados reais.

Além disso, precisamos tratar de maneira especial variaveis temporáis como o `mes_ref`, para evita que o modelo "veja o futuro". Uma alternativa bastante comum e discutida em estudos recentes, é a segmentação em variaveis ciclicas, como por exemplo:
`mes_ref` é decomposto em `ano` e `mes` (evita vazamento temporal direto).

"Apicella, Andrea, Francesco Isgrò, and Roberto Prevete. "Don’t push the button! exploring data leakage risks in machine learning and transfer learning." Artificial Intelligence Review 58.11 (2025): 1-58."

#### Preparacão inicial das features

In [4]:
df['mes_ref'] = pd.to_datetime(df['mes_ref'])
df['ano'] = df['mes_ref'].dt.year
df['mes'] = df['mes_ref'].dt.month

cols_drop = [c for c in ['id','documento','mes_ref'] if c in df.columns]
df_model = df.drop(columns=cols_drop)

X = df_model.drop(columns=['alvo'])
y = df_model['alvo'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

print(f"Shape treino/teste: {X_train.shape} / {X_test.shape}")
print(f"Numéricas: {len(num_cols)} | Categóricas: {len(cat_cols)}")


Shape treino/teste: (38185, 203) / (9547, 203)
Numéricas: 191 | Categóricas: 12


### <b>1.3.2 <span style='color:#2ae4f5'>|</span> Outliers</b>

Vou utilizar o Winsorizer (1%–99%) + RobustScaler para tratar os outliers (limitando-os ao valor máximo de q3 + 1.5 * iqr )

In [5]:
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
        self.bounds_ = None
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X, columns=self.feature_names_in_ if hasattr(self,'feature_names_in_') else None)
        lows = X_df.quantile(self.lower)
        highs = X_df.quantile(self.upper)
        self.bounds_ = pd.DataFrame({'low': lows, 'high': highs})
        return self
    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names_in_ if hasattr(self,'feature_names_in_') else None)
        for c in X_df.columns:
            low = self.bounds_.loc[c,'low']
            high = self.bounds_.loc[c,'high']
            X_df[c] = np.clip(X_df[c], low, high)
        return X_df.values

### <b>1.3.3 <span style='color:#2ae4f5'>|</span>  Valores Ausentes</b>

Sobre os valores ausentes, vou utilizar a abordagem mais comum inicialmente, que é:

* Imputação com mediana para valores numéricos.
* Imputação com valor mais frequente para categóricos.

In [6]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("winsor", Winsorizer(lower=0.01, upper=0.99)),
    ("scaler", RobustScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)

# <b>2 <span style='color:#00008B'>|</span>  Modelo</b>

## <b>1.1 <span style='color:#2ae4f5'>|</span>  Abordagem 1: Modelo XGBoost</b>

Em cenários como esse (detecção de fraude em concessão de crédito) estudos mostram que o XGBoost costuma apresentar os melhores resultados, uma vez que métricas de precisão costumam ser impactadas positivamente.

Por isso, vamos utilizar o modelo XGBoost como nossa primeira referência.  

#### Definição do modelo e tratamento:

Vamos definir o scale_pos_weight a partir do conjunto de treino 

In [35]:
pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = max(1.0, neg / max(1, pos))

O SMOTE será aplicado para lidar com o desbalanceamento das classes.

In [36]:
xgb = XGBClassifier(
    n_estimators=800,
    booster='gbtree',
    max_depth=4,
    importance_type='gain',
    gamma=0.3,
    learning_rate=0.015,
    subsample=1,
    colsample_bytree=0.6,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    eval_metric="auc",
    tree_method="exact",
    scale_pos_weight=scale_pos_weight,
    
)
pipe = ImbPipeline(steps=[
    ("prep", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", xgb)
])

Vamos usar **cross_val_predict** para obter **scores fora da amostra** (CV) e calibrar o **threshold**.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y_cv_proba = cross_val_predict(pipe, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:,1]

auc_cv = roc_auc_score(y_train, y_cv_proba)
ap_cv  = average_precision_score(y_train, y_cv_proba)

In [38]:
print(f"AUC-ROC (CV): {auc_cv:.4f} | AUC-PR (CV): {ap_cv:.4f}")

AUC-ROC (CV): 0.7617 | AUC-PR (CV): 0.3061


#### Threshold

A calibração do Threshold é extremamente importante em cenários como esse onde a precisão alta é desejada. Por isso, vamos definir o threshold que corresponda ao `ALERT_RATE`.

In [39]:
ALERT_RATE = 0.01   # 1% da base será alertada (ex.: operação quer revisar 1% com altíssima precisão)

In [ ]:
thr, prec_at_k, k = select_threshold_by_alert(y_train.reset_index(drop=True), pd.Series(y_cv_proba).reset_index(drop=True), ALERT_RATE)
print(f"Threshold escolhido: {thr:.4f} | Precision@Top{k} (~{ALERT_RATE*100:.1f}%) = {prec_at_k:.4f}")


Threshold escolhido: 0.9528 | Precision@Top381 (~1.0%) = 0.7428


Outra métrica importante de acompanhar em cenários de análise de crédito é o KS:

In [ ]:
ks_cv, fpr_cv, tpr_cv, thr_cv = ks_statistic(y_train, y_cv_proba)
print(f"KS (CV): {ks_cv:.4f}")


KS (CV): 0.3830


#### Treinamento e Resultados

Com isso, podemos treinar o modelo

In [42]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('winsor',
                                                                   Winsorizer()),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['vlr_financiado', 'VAR1',
                                                   'VAR2', 'VAR3', 'VAR4',
                                                   'VAR5', 'VAR6', 'VAR7',
                                                   'VAR8', 'VAR9', 'VAR10',
                                                   'VAR11', 'VAR12', 'VAR13',
                                                   'VAR14', 'VAR15', 'VAR16',
                                                   'VAR17', 'VAR18', 'VAR19',
                                                   'VAR20', 'VAR2...
                               feature_types=None, gamma=0.3, grow_policy=None,
                               importance_type='gain',
                               interaction_constraints=None,
                               learning_rate=0.015, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=4,
                               max_leaves=None, min_child_weight=None,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=800,
                               n_jobs=None, num_parallel_tree=None,
                               random_state=42, ...))])

E então análisar os resultados

In [ ]:
y_test_proba = pipe.predict_proba(X_test)[:,1]

auc_test = roc_auc_score(y_test, y_test_proba)
ap_test  = average_precision_score(y_test, y_test_proba)
ks_test, *_ = ks_statistic(y_test, y_test_proba)
print(f"[TEST] AUC-ROC: {auc_test:.4f} | AUC-PR: {ap_test:.4f} | KS: {ks_test:.4f}")

# Aplicar threshold calibrado pelo CV
y_test_pred = (y_test_proba >= thr).astype(int)
print("\n[TEST] Classification report")
print(classification_report(y_test, y_test_pred, digits=4))


[TEST] AUC-ROC: 0.7566 | AUC-PR: 0.2388 | KS: 0.3850

[TEST] Classification report (threshold calibrado para alta precisão)
              precision    recall  f1-score   support

           0     0.9636    0.9977    0.9803      9150
           1     0.7123    0.1310    0.2213       397

    accuracy                         0.9617      9547
   macro avg     0.8380    0.5643    0.6008      9547
weighted avg     0.9531    0.9617    0.9488      9547



### Análise do Impacto Financeiro

In [ ]:
vlr_test = df.loc[X_test.index, 'vlr_financiado'].values

resumo = simulate_financial_impact(y_test.values, y_test_proba, vlr_test, thr, PROFIT_MARGIN)
for k,v in resumo.items():
    if isinstance(v, (int,)):
        print(f"{k}: {v}")
    else:
        print(f"{k}: {v:,.4f}")


lucro_bons_aprovados (TN): 8,671,770.3930
perda_bons_bloqueados (FP): 24,281.6930
perda_fraudes_aprovadas (FN): 7,379,335.0000
ganho_fraudes_bloqueados (TP): 1,348,160.5500
impacto_liquido: 1,268,153.7000
precisao: 0.7123
recall: 0.1310
alerts: 73


O modelo apresentou uma boa acurácia e uma precisão interessante, mas o Recall está muito baixo (13.10%), o que indica que o modelo está capturando poucas fraudes reais. O KS apresentou um valor moderado (0.38), bom em comparação a aleatório, mas pode ser maior em problemas de fraude.
O impacto financeiro foi bastante interessante, com uma perda de propostas legitimas baixa, e um ganho em propostas fraudulentas expressivo. O problema é que ainda temos uma grande quantia que é fraudulenta e está sendo aprovada.

Por isso, tentaremos uma abordagem um pouco mais complexa. 

## <b>1.2 <span style='color:#2ae4f5'>|</span>  Abordagem 2: Stacked Ensemble</b>

Uma abordagem comum para problemas de fraude financeira é o Stacked Ensemble. Ele combina modelos de naturezas diferentes (ex: Random Forest, Logistic Regression, LightGBM, Redes Neurais, etc.) para que um meta-modelo aprenda quando confiar em cada um. A complexidade aumenta, o tempo de treino e tuning também se torcam mais custosos, mas pode ser uma abordagem interessante para aumentar a precisão e o recall em relação ao modelo XGBoost sozinho.

#### Definição dos modelos base e do meta:

In [8]:
base_lr = ('lr', LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    solver="saga",
    C=1.0,
    penalty="elasticnet",
    l1_ratio=0.5
))

base_rf = ('rf', RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
))

base_lgb = ('lgb', LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=128,
    max_depth=8,
    min_child_samples=10,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=round(len(y_train[y_train==0]) / len(y_train[y_train==1])),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    importance_type='gain'
))

base_cat = ('cat', CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=10,
    border_count=128,
    thread_count=-1,
    random_seed=RANDOM_STATE,
    verbose=False,
    auto_class_weights='Balanced'
))

final_estimator = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=round(len(y_train[y_train==0]) / len(y_train[y_train==1])),
    use_label_encoder=False,
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

#### Tratamentos e ajustes

O `StackingClassifier` não aceita aplicar SMOTE internamente para cada fold via `cross_val_predict`. Por isso, pra gerar **scores OOF** corretamente sem vazamento, precisei retirar o SMOTE do pipeline original e implementar um loop manual de `StratifiedKFold`.

Além disso, algumas alterações foram necessárias no pipeline. Foram elas:
- Substituí **SMOTE** simples por **SMOTEENN** (combinação de over + undersampling)
- Winsorização menos agressiva (0.5% vs 1%)
- StandardScaler em vez de RobustScaler para stacking
- Limitação de categorias no OneHotEncoder

In [9]:
class AdvancedWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.005, upper=0.995):
        self.lower = lower
        self.upper = upper
        self.bounds_ = None
    
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X, columns=self.feature_names_in_ if hasattr(self,'feature_names_in_') else None)
        lows = X_df.quantile(self.lower)
        highs = X_df.quantile(self.upper)
        self.bounds_ = pd.DataFrame({'low': lows, 'high': highs})
        return self
    
    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names_in_ if hasattr(self,'feature_names_in_') else None)
        for c in X_df.columns:
            low = self.bounds_.loc[c,'low']
            high = self.bounds_.loc[c,'high']
            X_df[c] = np.clip(X_df[c], low, high)
        return X_df.values

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('winsor', AdvancedWinsorizer(lower=0.005, upper=0.995)),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', max_categories=50))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, num_cols),
    ('cat', categorical_pipe, cat_cols)
], remainder='drop')


In [ ]:
def balanced_sampling_strategy(X, y, sampling_strategy='auto', k_neighbors=3):
    smote_enn = SMOTEENN(
        smote=SMOTE(sampling_strategy=0.3, k_neighbors=k_neighbors, random_state=RANDOM_STATE),
        enn=EditedNearestNeighbours(sampling_strategy='majority'),
        random_state=RANDOM_STATE
    )
    return smote_enn.fit_resample(X, y)


N_SPLITS = 7
oof_probas = np.zeros(len(X_train))
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f'Processing Fold {fold}/{N_SPLITS}...')
    
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    preprocessor.fit(X_tr, y_tr)
    X_tr_t = preprocessor.transform(X_tr)
    X_val_t = preprocessor.transform(X_val)
    
    X_tr_res, y_tr_res = balanced_sampling_strategy(X_tr_t, y_tr)
    
    estimators = [base_lr, base_rf, base_lgb, base_cat]
    
    stack = StackingClassifier(
        estimators=estimators, 
        final_estimator=final_estimator, 
        n_jobs=-1, 
        passthrough=False,
        cv=3
    )
    
    stack.fit(X_tr_res, y_tr_res)
    
    val_probas = stack.predict_proba(X_val_t)[:,1]
    oof_probas[val_idx] = val_probas

    fold_auc = roc_auc_score(y_val, val_probas)
    fold_ap = average_precision_score(y_val, val_probas)

auc_oof = roc_auc_score(y_train, oof_probas)
ap_oof = average_precision_score(y_train, oof_probas)

In [11]:
print(f'Fold {fold} - AUC: {fold_auc:.4f}, AP: {fold_ap:.4f}')
print(f'OOF Metrics - AUC: {auc_oof:.4f}, AP: {ap_oof:.4f}')

Fold 7 - AUC: 0.7194, AP: 0.2608
OOF Metrics - AUC: 0.7439, AP: 0.2721


#### Threshold

In [12]:
ALERT_RATE = 0.015   # 1.5% (um pequeno aumento para sermos menos conservador)

In [13]:
thr, prec_at_k, k = select_threshold_by_alert(
    y_train.reset_index(drop=True), 
    pd.Series(oof_probas).reset_index(drop=True), 
    ALERT_RATE
)
print(f'Selected threshold: {thr:.4f} | Precision@Top{k} = {prec_at_k:.4f} (Alert rate {ALERT_RATE*100:.2f}%)')


Selected threshold: 0.9845 | Precision@Top572 = 0.6014 (Alert rate 1.50%)


#### Treinamento e Resultados

In [ ]:
preprocessor.fit(X_train, y_train)
X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)

X_train_res, y_train_res = balanced_sampling_strategy(X_train_t, y_train)
estimators = [base_lr, base_rf, base_lgb, base_cat]
stack_final = StackingClassifier(
    estimators=estimators, 
    final_estimator=final_estimator, 
    n_jobs=-1, 
    passthrough=False,
    cv=3
)

stack_final.fit(X_train_res, y_train_res)

y_test_proba = stack_final.predict_proba(X_test_t)[:,1]

In [15]:
print('Test AUC:', roc_auc_score(y_test, y_test_proba))
print('Test AP :', average_precision_score(y_test, y_test_proba))

ks_test, fpr_t, tpr_t, thr_t = (lambda y, s: (np.max(roc_curve(y,s)[1] - roc_curve(y,s)[0]),) + tuple(roc_curve(y,s)))(y_test, y_test_proba)
print('Test KS :', ks_test)

Test AUC: 0.7129358164374888
Test AP : 0.2056217385943127
Test KS : 0.32512284758640625


Vamos analisar o impacto de diferentes Trhesholds:

In [16]:
test_thresholds = [
    np.percentile(y_test_proba, 98.5),  # Top 1.5%
    np.percentile(y_test_proba, 98.0),  # Top 2%
    np.percentile(y_test_proba, 97.5),  # Top 2.5%
    thr  # O Threshold otimizado
]

for i, test_thr in enumerate(test_thresholds):
    y_test_pred = (y_test_proba >= test_thr).astype(int)
    
    alerts = int((y_test_proba >= test_thr).sum())
    alert_rate = alerts / len(y_test) * 100
    
    prec_test = precision_score(y_test, y_test_pred)
    recall_test = recall_score(y_test, y_test_pred)
    f1_test = f1_score(y_test, y_test_pred)
    
    print(f'\nThreshold {i+1}: {test_thr:.4f} (Alert Rate: {alert_rate:.2f}%)')
    print(f'Precision: {prec_test:.4f}, Recall: {recall_test:.4f}, F1: {f1_test:.4f}')
    
    # Impacto financeiro
    vlr_test = df.loc[X_test.index, 'vlr_financiado'].values
    resumo = simulate_financial_impact(y_test.values, y_test_proba, vlr_test, test_thr, PROFIT_MARGIN)
    print(f'Impacto Líquido: {resumo["impacto_liquido"]:,.2f}')

final_threshold = thr 

y_test_pred_final = (y_test_proba >= final_threshold).astype(int)
print("\n[TEST] Classification report")
print(classification_report(y_test, y_test_pred_final, digits=4))


Threshold 1: 0.9176 (Alert Rate: 1.51%)
Precision: 0.4722, Recall: 0.1713, F1: 0.2514
Impacto Líquido: 1,549,944.21

Threshold 2: 0.7799 (Alert Rate: 2.00%)
Precision: 0.3770, Recall: 0.1814, F1: 0.2449
Impacto Líquido: 1,556,123.36

Threshold 3: 0.5726 (Alert Rate: 2.50%)
Precision: 0.3138, Recall: 0.1889, F1: 0.2358
Impacto Líquido: 1,516,026.25

Threshold 4: 0.9845 (Alert Rate: 1.09%)
Precision: 0.5962, Recall: 0.1562, F1: 0.2475
Impacto Líquido: 1,477,419.04

[TEST] Classification report
              precision    recall  f1-score   support

           0     0.9645    0.9954    0.9797      9150
           1     0.5962    0.1562    0.2475       397

    accuracy                         0.9605      9547
   macro avg     0.7803    0.5758    0.6136      9547
weighted avg     0.9492    0.9605    0.9493      9547



#### Análise do Impacto Financeiro

In [17]:
resumo_final = simulate_financial_impact(y_test.values, y_test_proba, vlr_test, final_threshold, PROFIT_MARGIN)
for k, v in resumo_final.items():
    if isinstance(v, (int,)):
        print(f'{k}: {v}')
    else:
        print(f'{k}: {v:,.4f}')

lucro_bons_aprovados (TN): 8,649,437.8455
perda_bons_bloqueados (FP): 46,614.2405
perda_fraudes_aprovadas (FN): 7,125,404.5600
ganho_fraudes_bloqueados (TP): 1,602,090.9900
impacto_liquido: 1,477,419.0450
precisao: 0.5962
recall: 0.1562
alerts: 104
